# Football club stocks and betting-market surprises

**Hypothesis.** Listed football club shares underreact to match results measured against what the betting market expected: after a surprise, the price keeps drifting in the direction of the surprise for several trading days.

This notebook runs the whole pipeline in the order of the report: **data acquisition → cleaning → strategy → backtest**. The code lives in `src/` (one module per step); each step's design, decisions and checks are written up in `docs/` (`docs/data_sources.md` lists every source and how it was acquired).

**How to run.** Run all cells: the first cell installs the packages in `requirements.txt` (Python 3.11 or later). With `DOWNLOAD = False` (default) every step reads the raw data snapshot in `data/raw/` and rebuilds `data/processed/` and `figures/` from it, in about two minutes. With `DOWNLOAD = True` the acquisition cells fetch whatever is missing from `data/raw/` (files on disk are never re-downloaded or changed). The Betfair download needs a Betfair account in `.env` (see `.env.example`); no credentials are in this repository. Licensed raw data (Betfair, OddsPortal, Bloomberg) is not included either: `data/README.md` says how to get every source.

### Setup
Moves to the repository root (Jupyter often starts in `notebooks/`), then installs the tested package versions into the environment this notebook runs in (`requirements.txt`; needs network the first time, then does nothing). If any of them were already loaded in this session at another version, restart the kernel and run all again.

In [ ]:
import os
from pathlib import Path

if Path.cwd().name == "notebooks":
    os.chdir("..")
%pip install -q -r requirements.txt

In [ ]:
from pathlib import Path
import os
import pandas as pd
from IPython.display import Image, display

from src.common import PROCESSED, ROOT, load_config

DOWNLOAD = False          # True: fetch missing raw files (network; Betfair needs .env credentials)
RAW = ROOT / "data" / "raw"
FIGURES = ROOT / "figures"
pd.set_option("display.width", 160, "display.max_columns", 30)
cfg = load_config()
cfg.clubs[["club", "league", "ticker", "exchange", "benchmark"]]

## 1. Data acquisition

| Source | What it gives | How |
|---|---|---|
| football-data.co.uk | Domestic league results and closing odds, 2012/13 to 2025/26 | Public CSV download |
| Betfair Exchange historical data (Basic) | Pre-kickoff prices, results and kickoff times of every match of the clubs, May 2015 to May 2026 | Authenticated API (`betfairlightweight`), credentials in `.env` |
| OddsPortal | Odds and results Betfair lacks (2012/13 to 2014/15 European and cup matches, gaps after 2015), competition names | OddsHarvester, a separate scraping tool (commands in `docs/step1b_betfair.md`) |
| openfootball | European fixture list (competition, date) | Public GitHub files |
| Wikipedia (club and league season articles) | Competition names, friendlies, kickoff times, coverage check; league tables for the stakes split | MediaWiki API, 3 s between requests |
| Yahoo Finance | Daily prices of 13 club stocks and the benchmark indices | `yfinance` |
| Bloomberg (Imperial library terminal) | AS Roma (delisted 2022), PSI 20 index, corporate action factors | Manual BDH export to Excel, documented in `docs/bloomberg_pull_2026-10-08.md` |

The universe is the 14 European clubs whose shares were listed during 2012 to 2026 (`config/clubs.yaml`), including Roma, which was delisted in 2022.

In [ ]:
def inventory(raw=RAW):
    """Files and size per raw source in the snapshot."""
    rows = []
    for d in sorted(p for p in raw.iterdir() if p.is_dir()):
        files = [f for f in d.rglob("*") if f.is_file()]
        rows.append({"source": d.name, "files": len(files), "MB": round(sum(f.stat().st_size for f in files) / 1e6, 1)})
    return pd.DataFrame(rows)

inventory()

### 1.1 League results and odds (football-data.co.uk)
One CSV per league and season, saved unmodified.

In [ ]:
from src.step1_expectations import download_season

if DOWNLOAD:
    for league in cfg.leagues:
        for season in cfg.seasons:
            download_season(league, season)
sorted(p.name for p in (RAW / "football_data").glob("*.csv"))[:5], len(list((RAW / "football_data").glob("*.csv")))

### 1.2 European and cup odds (Betfair Exchange, OddsPortal)
Betfair files are searched month by month by club name and kept when a team matches a configured alias. OddsPortal pages were fetched once with OddsHarvester for what Betfair lacks; they cannot be re-fetched from this notebook.

In [ ]:
from src.step1b_betfair import download_basic

if DOWNLOAD and ((ROOT / ".env").exists() or "BETFAIR_APP_KEY" in os.environ):
    download_basic("2015-05-01", "2026-05-31")   # about 80 minutes; resumable
print("Betfair files:", sum(1 for _ in (RAW / "betfair").rglob("*.bz2")))
print("OddsPortal files:", sum(1 for f in (RAW / "oddsportal").rglob("*") if f.is_file()))

### 1.3 Fixtures and tables (openfootball, Wikipedia)

In [ ]:
from src.european_fixtures import download_openfootball
from src.wikipedia_fixtures import download_seasons
from src.league_tables import download_tables

if DOWNLOAD:
    download_openfootball([f"{y}-{(y + 1) % 100:02d}" for y in range(2012, 2026)])
    download_seasons(2012, 2025)      # club season articles, about 15 minutes
    download_tables()                 # league season articles, about 7 minutes
inventory().query("source in ['openfootball', 'wikipedia', 'wikipedia_leagues']")

### 1.4 Share prices (Yahoo Finance, Bloomberg)
Yahoo: open, high, low, close, adjusted close, volume, dividends and splits from 2012, saved unmodified. Bloomberg: Roma (no Yahoo history after delisting) and the PSI 20 (Yahoo's starts in 2013), exported by hand from the library terminal (`data/raw/bloomberg/`).

In [ ]:
from src.step3_prices import download_prices

if DOWNLOAD:
    download_prices()
sorted(f.name for f in (RAW / "prices").glob("*.csv")) + sorted(f.name for f in (RAW / "bloomberg").glob("*.xlsx"))

## 2. Cleaning

From raw files to one row per club per match with the market's pre-match probabilities, the result after 90 minutes, the kickoff time and the competition; then share prices with corporate actions fixed, trading calendars, and the first trading day each result can affect (event day 0). Every dropped or corrected row is logged (`data/processed/*.csv`, `docs/issues.md`); the checks print below.

### 2.1 League matches: odds to probabilities
Closing odds (Pinnacle, else market average, else Betfair) converted to probabilities by removing the bookmaker margin proportionally. Forfeits and non-standard seasons are listed in the config.

In [ ]:
from src.step1_expectations import build_match_expectations

league = build_match_expectations(download=False)

### 2.2 European, cup and super cup matches
Betfair last traded prices before kickoff; OddsPortal where Betfair has no match; competitions named from OddsPortal listings and Wikipedia; friendlies dropped; results after 90 minutes (extra time handled); kickoff times from Betfair, OddsPortal or Wikipedia.

In [ ]:
from src.step1b_betfair import build_betfair_expectations

all_matches = build_betfair_expectations()

### 2.3 Share prices: corporate actions, trading calendars, event day 0
Yahoo misdates seven Turkish rights issues and one Juventus rights issue; they are corrected with Bloomberg's adjustment factors. Trading calendars count only days with volume. Event day 0 is the first trading day whose close comes after the estimated end of the match (kickoff + 115 minutes) in the exchange's time zone.

In [ ]:
from src.step2_surprise import build_match_surprise
from src.step3_prices import build_match_events

surprise = build_match_surprise()     # needed by Step 3; the surprise itself is the signal (section 3.1)
events = build_match_events()

### 2.4 Abnormal returns
Daily return minus beta times the local index return; beta is a Dimson beta (market returns of the day before, the day and the day after, to handle stale prices), estimated on trading days t-260 to t-11, so it never uses the event window.

In [ ]:
from src.step4_event_study import build_abnormal_returns

panel = build_abnormal_returns()

### 2.5 League tables (for the stakes split)
Tables on the morning of every league match, built from football-data results with point deductions; places and checks from Wikipedia's final tables.

In [ ]:
from src.league_tables import build_e1

stakes = build_e1()

## 3. Strategy

### 3.1 The signal: match surprise
`surprise = actual score − expected score`, with the score +1 for a win, 0 for a draw, −1 for a loss and the expected score `p_win − p_loss` from the betting market. It runs from −2 to +2 and is 0 when the result matched expectations on average. Matches are put in terciles (`low`, `mid`, `high`).

In [ ]:
s = pd.read_csv(PROCESSED / "match_surprise.csv")
display(s.groupby("competition_type")["surprise"].describe().round(3))
s.groupby("surprise_tercile")["surprise"].agg(["count", "min", "max", "mean"]).round(3)

### 3.2 Does the price keep drifting after a surprise? (event study)
A club plays every 3 to 4 trading days, so fixed event windows overlap the next match. The main test is a panel regression of each club-day abnormal return on the surprises of all the club's matches at lags −3 to +10 trading days. Pre-specified tests: **H1** drift (sum of the lag 1 to 3 coefficients), **H2** good v bad news, **H3** league v European v cup, with a Holm correction across the three; robustness variants and a placebo with shuffled surprises.

In [ ]:
from src.step4_event_study import run_event_study

tests = run_event_study()

In [ ]:
display(Image(FIGURES / "event_study_cumulative_response.png"))
display(Image(FIGURES / "event_study_clean_car.png"))

### 3.3 Exploratory splits and the match-week pattern
Not pre-specified tests: stakes (E1), form (E2), period (E3), surprise size (A1), weekend gap (A2); and the abnormal return around every match whatever its result.

In [ ]:
from src.step4_event_study import run_exploratory, run_match_week

exploratory = run_exploratory()
week = run_match_week()
display(Image(FIGURES / "match_week_pattern.png"))

## 4. Backtest

Rules (fixed in `docs/step4_event_study.md` before any backtest was run):
- **Signal:** the match's surprise tercile, with cutoffs from earlier seasons only (walk-forward); 2012/13 is the burn-in, trading from 2013/14.
- **Trade:** long after a `high` surprise, short after a `low` one, entry at the close of day 0, exit at the close of day +3 or before the club's next match, whichever is first; each position hedged with beta × the local index.
- **Portfolio:** equal weights across open positions (calendar time).
- **Costs:** half the estimated bid-ask spread per side (Corwin–Schultz; Abdi–Ranaldo for the two stocks that mostly trade once a day) plus 0.10% commission per side and transaction taxes.
- **In-sample** 2013/14 to 2018/19, **out-of-sample** 2019/20 to 2025/26.

In [ ]:
from src.step4_event_study import run_backtest

summary = run_backtest()

### Key results

In [ ]:
key = summary.pivot_table(index=["version", "costs"], columns="period",
                          values=["cumulative_return", "sharpe", "max_drawdown"], sort=False)
key = key.reindex(columns=["in_sample", "out_of_sample", "full"], level=1)
key.round(3)

In [ ]:
display(Image(FIGURES / "backtest_cumulative.png"))

### Charts for the write up
Dark theme versions of the backtest and the match-week pattern (`src/substack_charts.py`).

In [ ]:
from src.substack_charts import charts

charts()
display(Image(FIGURES / "substack_backtest.png"))
display(Image(FIGURES / "substack_matchweek.png"))